# Mamba vs Transformer Comparison in Tiny Recursive Model


In [ ]:
import os
import sys
os.environ["MAMBA_SELECTIVE_SCAN"] = "0"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"

## Config for choosing dataset

In [ ]:
doARCAGI = True
doSudoku = False
doMaze = False

doFullTests = True
doQuickTests = False

### Dependencies

In [ ]:
%pip install mambapy einops

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
%pip install arckit matplotlib pandas tensorboard huggingface_hub

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


## Transformer and Mamba blocks

In [ ]:
import time
import json
from collections import defaultdict
from typing import Tuple, List, Dict, Optional, Union
from dataclasses import dataclass
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from torch import optim
from torch.nn.functional import scaled_dot_product_attention
from torch.utils.tensorboard import SummaryWriter
from pydantic import BaseModel
import einops
import warnings
warnings.filterwarnings('ignore')
import arckit

try:
    from mambapy.mamba import Mamba, MambaConfig
    MAMBA2_AVAILABLE = True
except ImportError:
    MAMBA2_AVAILABLE = False

IGNORE_LABEL_ID = -100

def trunc_normal_init_(tensor: torch.Tensor, std: float = 1.0, lower: float = -2.0, upper: float = 2.0):
    with torch.no_grad():
        if std == 0:
            tensor.zero_()
        else:
            sqrt2 = math.sqrt(2)
            a = math.erf(lower / sqrt2)
            b = math.erf(upper / sqrt2)
            z = (b - a) / 2
            c = (2 * math.pi) ** -0.5
            pdf_u = c * math.exp(-0.5 * lower ** 2)
            pdf_l = c * math.exp(-0.5 * upper ** 2)
            comp_std = std / math.sqrt(1 - (upper * pdf_u - lower * pdf_l) / z - ((pdf_u - pdf_l) / z) ** 2)
            tensor.uniform_(a, b)
            tensor.erfinv_()
            tensor.mul_(sqrt2 * comp_std)
            tensor.clip_(lower * comp_std, upper * comp_std)
    return tensor

CosSin = Tuple[torch.Tensor, torch.Tensor]

def _find_multiple(a, b):
    return (-(a // -b)) * b

def rotate_half(x: torch.Tensor):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    orig_dtype = q.dtype
    q = q.to(cos.dtype)
    k = k.to(cos.dtype)
    q_embed = (q * cos.unsqueeze(-2)) + (rotate_half(q) * sin.unsqueeze(-2))
    k_embed = (k * cos.unsqueeze(-2)) + (rotate_half(k) * sin.unsqueeze(-2))
    return q_embed.to(orig_dtype), k_embed.to(orig_dtype)

class CastedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool):
        super().__init__()
        self.weight = nn.Parameter(
            trunc_normal_init_(torch.empty((out_features, in_features)), std=1.0 / (in_features ** 0.5))
        )
        self.bias = None
        if bias:
            self.bias = nn.Parameter(torch.zeros((out_features, )))

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.linear(input, self.weight.to(input.dtype), bias=self.bias.to(input.dtype) if self.bias is not None else None)

class CastedEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, init_std: float, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to
        self.embedding_weight = nn.Parameter(
            trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std)
        )

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.embedding(input, self.embedding_weight.to(self.cast_to))

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings, base, device=None):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32, device=device) / dim))
        t = torch.arange(max_position_embeddings, dtype=torch.float32, device=device)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)

    def forward(self):
        return self.cos_cached, self.sin_cached

class Attention(nn.Module):
    def __init__(self, hidden_size, head_dim, num_heads, num_key_value_heads, causal=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.head_dim = head_dim
        self.output_size = head_dim * num_heads
        self.num_heads = num_heads
        self.num_key_value_heads = num_key_value_heads
        self.causal = causal
        self.qkv_proj = CastedLinear(self.hidden_size, (self.num_heads + 2 * self.num_key_value_heads) * self.head_dim, bias=False)
        self.o_proj = CastedLinear(self.output_size, self.hidden_size, bias=False)

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape
        qkv = self.qkv_proj(hidden_states)
        qkv = qkv.view(batch_size, seq_len, self.num_heads + 2 * self.num_key_value_heads, self.head_dim)
        query = qkv[:, :, :self.num_heads]
        key = qkv[:, :, self.num_heads: self.num_heads + self.num_key_value_heads]
        value = qkv[:, :, self.num_heads + self.num_key_value_heads:]
        if cos_sin is not None:
            cos, sin = cos_sin
            query, key = apply_rotary_pos_emb(query, key, cos, sin)
        query, key, value = map(lambda t: einops.rearrange(t, 'B S H D -> B H S D'), (query, key, value))
        attn_output = scaled_dot_product_attention(query=query, key=key, value=value, is_causal=self.causal)
        attn_output = einops.rearrange(attn_output, 'B H S D -> B S H D')
        attn_output = attn_output.view(batch_size, seq_len, self.output_size)
        return self.o_proj(attn_output)

class SwiGLU(nn.Module):
    def __init__(self, hidden_size: int, expansion: float):
        super().__init__()
        inter = _find_multiple(round(expansion * hidden_size * 2 / 3), 256)
        self.gate_up_proj = CastedLinear(hidden_size, inter * 2, bias=False)
        self.down_proj = CastedLinear(inter, hidden_size, bias=False)

    def forward(self, x):
        gate, up = self.gate_up_proj(x).chunk(2, dim=-1)
        return self.down_proj(F.silu(gate) * up)

def rms_norm(hidden_states: torch.Tensor, variance_epsilon: float) -> torch.Tensor:
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)
    variance = hidden_states.square().mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return hidden_states.to(input_dtype)

class CastedSparseEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, batch_size: int, init_std: float, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to
        self.register_buffer('weights', trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std), persistent=True)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch_size = inputs.shape[0]
        if not self.training:
            return self.weights[inputs].to(self.cast_to)
        local_weights = self.weights[inputs].clone().requires_grad_(True)
        return local_weights.to(self.cast_to)

class Mamba2Block(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, norm_eps=1e-5):
        super().__init__()
        if not MAMBA2_AVAILABLE:
            raise ImportError("mambapy is required for Mamba2Block")
        try:
            config = MambaConfig(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                n_layers=1,
            )
            self.mamba = Mamba(config)
        except Exception as e:
            raise RuntimeError(f"Failed to initialize Mamba: {e}")

    def forward(self, x):
        if x.dim() != 3:
            raise ValueError(f"Mamba2Block expects 3D input [batch, seq, dim], got {x.dim()}D")
        return self.mamba(x)

@dataclass
class TinyRecursiveReasoningModel_ACTV1InnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor

@dataclass
class TinyRecursiveReasoningModel_ACTV1Carry:
    inner_carry: TinyRecursiveReasoningModel_ACTV1InnerCarry
    steps: torch.Tensor
    halted: torch.Tensor
    current_data: Dict[str, torch.Tensor]

class TinyRecursiveReasoningModel_ACTV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int
    H_cycles: int
    L_cycles: int
    H_layers: int
    L_layers: int
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str
    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    halt_max_steps: int
    halt_exploration_prob: float
    forward_dtype: str = "bfloat16"
    mlp_t: bool = False
    use_mamba2: bool = False
    puzzle_emb_len: int = 16
    no_ACT_continue: bool = True

class TinyRecursiveReasoningModel_ACTV1Block(nn.Module):
    def __init__(self, config: TinyRecursiveReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        if self.config.mlp_t:
            self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size) if self.config.puzzle_emb_len == 0 else self.config.puzzle_emb_len
            self.mlp_t = SwiGLU(
                hidden_size=self.config.seq_len + self.puzzle_emb_len,
                expansion=config.expansion,
            )
        elif self.config.use_mamba2:
            mamba_d_state = getattr(config, 'mamba_d_state', 16)
            mamba_d_conv = getattr(config, 'mamba_d_conv', 4)
            self.mamba2 = Mamba2Block(
                d_model=config.hidden_size,
                d_state=mamba_d_state,
                d_conv=mamba_d_conv,
                expand=int(config.expansion)
            )
        else:
            self.self_attn = Attention(
                hidden_size=config.hidden_size,
                head_dim=config.hidden_size // config.num_heads,
                num_heads=config.num_heads,
                num_key_value_heads=config.num_heads,
                causal=False
            )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        if self.config.mlp_t:
            hidden_states = hidden_states.transpose(1,2)
            out = self.mlp_t(hidden_states)
            hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
            hidden_states = hidden_states.transpose(1,2)
        elif self.config.use_mamba2:
            hidden_states = rms_norm(hidden_states + self.mamba2(hidden_states), variance_epsilon=self.norm_eps)
        else:
            hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        out = self.mlp(hidden_states)
        hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
        return hidden_states

class TinyRecursiveReasoningModel_ACTV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[TinyRecursiveReasoningModel_ACTV1Block]):
        super().__init__()
        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, **kwargs) -> torch.Tensor:
        hidden_states = hidden_states + input_injection
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)
        return hidden_states

class TinyRecursiveReasoningModel_ACTV1_Inner(nn.Module):
    def __init__(self, config: TinyRecursiveReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)
        self.embed_scale = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale
        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head = CastedLinear(self.config.hidden_size, 2, bias=True)
        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size) if self.config.puzzle_emb_len == 0 else self.config.puzzle_emb_len
        if self.config.puzzle_emb_ndim > 0:
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)
        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.L_level = TinyRecursiveReasoningModel_ACTV1ReasoningModule(layers=[TinyRecursiveReasoningModel_ACTV1Block(self.config) for _i in range(self.config.L_layers)])
        H_init_tensor = trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1)
        L_init_tensor = trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1)
        self.register_buffer('H_init', H_init_tensor, persistent=True)
        self.register_buffer('L_init', L_init_tensor, persistent=True)
        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        embedding = self.embed_tokens(input.to(torch.int32))
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))
            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)
        if self.config.pos_encodings == "learned":
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))
        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int):
        device = next(self.parameters()).device
        return TinyRecursiveReasoningModel_ACTV1InnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype, device=device),
            z_L=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype, device=device),
        )

    def reset_carry(self, reset_flag: torch.Tensor, carry: TinyRecursiveReasoningModel_ACTV1InnerCarry):
        batch_size = carry.z_H.shape[0]
        seq_len = carry.z_H.shape[1]
        device = carry.z_H.device
        reset_flag = reset_flag.to(device)
        H_init_tensor = self.H_init.to(device)
        L_init_tensor = self.L_init.to(device)
        H_init_expanded = H_init_tensor.unsqueeze(0).unsqueeze(0).expand(batch_size, seq_len, -1)
        L_init_expanded = L_init_tensor.unsqueeze(0).unsqueeze(0).expand(batch_size, seq_len, -1)
        return TinyRecursiveReasoningModel_ACTV1InnerCarry(
            z_H=torch.where(reset_flag.view(-1, 1, 1), H_init_expanded, carry.z_H),
            z_L=torch.where(reset_flag.view(-1, 1, 1), L_init_expanded, carry.z_L),
        )

    def forward(self, carry: TinyRecursiveReasoningModel_ACTV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[TinyRecursiveReasoningModel_ACTV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )
        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])
        z_H, z_L = carry.z_H, carry.z_L
        with torch.no_grad():
            for _H_step in range(self.config.H_cycles-1):
                for _L_step in range(self.config.L_cycles):
                    z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
                z_H = self.L_level(z_H, z_L, **seq_info)
        for _L_step in range(self.config.L_cycles):
            z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.L_level(z_H, z_L, **seq_info)
        new_carry = TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())
        output = self.lm_head(z_H)[:, self.puzzle_emb_len:]
        q_logits = self.q_head(z_H[:, 0]).to(torch.float32)
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])

class TinyRecursiveReasoningModel_ACTV1(nn.Module):
    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = TinyRecursiveReasoningModel_ACTV1Config(**config_dict)
        self.inner = TinyRecursiveReasoningModel_ACTV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]
        device = batch["inputs"].device
        return TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),
            steps=torch.zeros((batch_size, ), dtype=torch.int32, device=device),
            halted=torch.ones((batch_size, ), dtype=torch.bool, device=device),
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )

    def forward(self, carry: TinyRecursiveReasoningModel_ACTV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[TinyRecursiveReasoningModel_ACTV1Carry, Dict[str, torch.Tensor]]:
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        new_steps = torch.where(carry.halted, 0, carry.steps)
        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}
        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)
        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }
        with torch.no_grad():
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            halted = is_last_step
            if self.training and (self.config.halt_max_steps > 1):
                if self.config.no_ACT_continue:
                    halted = halted | (q_halt_logits > 0)
                else:
                    halted = halted | (q_halt_logits > q_continue_logits)
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(new_steps, low=2, high=self.config.halt_max_steps + 1)
                halted = halted & (new_steps >= min_halt_steps)
                if not self.config.no_ACT_continue:
                    _, _, (next_q_halt_logits, next_q_continue_logits) = self.inner(new_inner_carry, new_current_data)
                    outputs["target_q_continue"] = torch.sigmoid(torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits)))
        return TinyRecursiveReasoningModel_ACTV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs

class MambaBiDirectionalModel(nn.Module):
    def __init__(self, config_dict: dict):
        super().__init__()
        config_dict_1 = config_dict.copy()
        config_dict_2 = config_dict.copy()
        self.config = TinyRecursiveReasoningModel_ACTV1Config(**config_dict)
        self.model_1 = TinyRecursiveReasoningModel_ACTV1(config_dict_1)
        self.model_2 = TinyRecursiveReasoningModel_ACTV1(config_dict_2)

    @property
    def puzzle_emb(self):
        return self.model_1.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        carry_1 = self.model_1.initial_carry(batch)
        carry_2 = self.model_2.initial_carry(batch)
        return (carry_1, carry_2)

    def forward(self, carry_tuple, batch: Dict[str, torch.Tensor]) -> Tuple[Tuple, Dict[str, torch.Tensor]]:
        carry_1, carry_2 = carry_tuple

        batch_reversed = {k: v.flip(dims=[1]) if k == "inputs" else v for k, v in batch.items()}

        carry_1, outputs_1 = self.model_1(carry_1, batch)
        carry_2, outputs_2 = self.model_2(carry_2, batch_reversed)

        logits_1 = outputs_1["logits"]
        logits_2 = outputs_2["logits"].flip(dims=[1])

        combined_logits = torch.cat([logits_1, logits_2], dim=-1)

        combined_outputs = {
            "logits": combined_logits,
            "q_halt_logits": outputs_1["q_halt_logits"],
            "q_continue_logits": outputs_1["q_continue_logits"]
        }

        return (carry_1, carry_2), combined_outputs


2025-11-20 15:54:41.531347: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-20 15:54:42.390304: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [ ]:
#configs for transformer and mamba

transformer_config = {
    "batch_size": 1,
    "seq_len": 3800,
    "num_puzzle_identifiers": 1000,
    "vocab_size": 12,
    "H_cycles": 3,
    "L_cycles": 2,
    "H_layers": 4,
    "L_layers": 2,
    "hidden_size": 384,
    "expansion": 2.0,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 8,
    "halt_exploration_prob": 0.1,
    "use_mamba2": False,
    "mamba_d_state": 16,
    "mamba_d_conv": 4,
    "puzzle_emb_ndim": 256,
    "forward_dtype": "bfloat16",
    "no_ACT_continue": True
}

mamba_config = {
    "batch_size": 1,
    "seq_len": 3800,
    "num_puzzle_identifiers": 1000,
    "vocab_size": 12,
    "H_cycles": 3,
    "L_cycles": 2,
    "H_layers": 6,
    "L_layers": 3,
    "hidden_size": 384,
    "expansion": 2.0,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 8,
    "halt_exploration_prob": 0.1,
    "use_mamba2": True,
    "mamba_d_state": 16,
    "mamba_d_conv": 4,
    "puzzle_emb_ndim": 256,
    "forward_dtype": "bfloat16",
    "no_ACT_continue": True
}

mamba_bidirectional_config = {
    "batch_size": 1,
    "seq_len": 3800,
    "num_puzzle_identifiers": 1000,
    "vocab_size": 12,
    "H_cycles": 3,
    "L_cycles": 2,
    "H_layers": 3,
    "L_layers": 1,
    "hidden_size": 320,
    "expansion": 2.0,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 8,
    "halt_exploration_prob": 0.1,
    "use_mamba2": True,
    "mamba_d_state": 16,
    "mamba_d_conv": 4,
    "puzzle_emb_ndim": 256,
    "forward_dtype": "bfloat16",
    "no_ACT_continue": True
}

# ARC-AGI
## ARCReasoningDatasetClass

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from typing import Dict, Tuple

IGNORE_LABEL_ID = -100

class ARCReasoningDataset(Dataset):
    def __init__(self, arc_data, seq_len: int = 4500, pad_token_id: int = 0):
        self.arc_data = arc_data
        self.seq_len = seq_len
        self.pad_token_id = pad_token_id
        self.puzzle_to_id = {}
        self._build_puzzle_mapping()

    def _build_puzzle_mapping(self):
        for idx, puzzle in enumerate(self.arc_data):
            self.puzzle_to_id[puzzle.id] = idx

    def _create_reasoning_sequence(self, puzzle) -> Tuple[torch.Tensor, torch.Tensor]:
        context_parts = []
        for demo in puzzle.train:
            if isinstance(demo, tuple):
                input_grid, output_grid = demo
            else:
                input_grid = demo.input
                output_grid = demo.output

            input_flat = input_grid.flatten()
            output_flat = output_grid.flatten()

            demo_seq = np.concatenate([
                input_flat + 1,
                output_flat + 1,
                [11]  # separator
            ])
            context_parts.append(demo_seq)

        context = np.concatenate(context_parts)

        # Handle various test formats
        if isinstance(puzzle.test, list):
            test_example = puzzle.test[0]
            if isinstance(test_example, tuple):
                test_input_grid, test_output_grid = test_example
            else:
                test_input_grid = test_example.input
                test_output_grid = test_example.output
        elif isinstance(puzzle.test, tuple):
            test_input_grid, test_output_grid = puzzle.test
        else:
            test_input_grid = puzzle.test.input
            test_output_grid = puzzle.test.output

        test_input = test_input_grid.flatten() + 1

        full_input = np.concatenate([context, test_input])
        full_target = np.concatenate([
            np.full_like(context, IGNORE_LABEL_ID),
            test_output_grid.flatten() + 1
        ])

        input_tensor = torch.from_numpy(full_input).long()
        target_tensor = torch.from_numpy(full_target).long()

        # --- NEW: robust fixed-length packing ---
        seq_len = self.seq_len

        fixed_inputs = torch.full(
            (seq_len,),
            self.pad_token_id,
            dtype=torch.long
        )
        fixed_targets = torch.full(
            (seq_len,),
            IGNORE_LABEL_ID,
            dtype=torch.long
        )

        L_inp = min(seq_len, input_tensor.size(0))
        L_tgt = min(seq_len, target_tensor.size(0))

        fixed_inputs[:L_inp] = input_tensor[:L_inp]
        fixed_targets[:L_tgt] = target_tensor[:L_tgt]
        # --- end new block ---

        return fixed_inputs, fixed_targets

    def __len__(self):
        return len(self.arc_data)

    def __getitem__(self, idx) -> Dict[str, torch.Tensor]:
        puzzle = self.arc_data[idx]
        inputs, targets = self._create_reasoning_sequence(puzzle)

        # these should now always be true
        assert inputs.size(0) == self.seq_len
        assert targets.size(0) == self.seq_len

        return {
            "inputs": inputs,
            "targets": targets,
            "labels": targets.clone(),
            "puzzle_identifiers": torch.tensor(
                self.puzzle_to_id[puzzle.id],
                dtype=torch.long
            )
        }


def create_arc_reasoning_dataloaders(
    train_data,
    eval_data,
    batch_size: int = 2,
    seq_len: int = 4500,
    num_workers: int = 0,
) -> Tuple[DataLoader, DataLoader]:

    train_dataset = ARCReasoningDataset(train_data, seq_len=seq_len)
    eval_dataset = ARCReasoningDataset(eval_data, seq_len=seq_len)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, eval_loader


In [ ]:
#test correct size for arc_reasoning_dataloader

train_data, eval_data = arckit.load_data("kaggle")
train_loader, eval_loader = create_arc_reasoning_dataloaders(
    train_data,
    eval_data,
    batch_size=2,
    seq_len=4500,
)

batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape, v.dtype)


inputs torch.Size([2, 4500]) torch.int64
targets torch.Size([2, 4500]) torch.int64
labels torch.Size([2, 4500]) torch.int64
puzzle_identifiers torch.Size([2]) torch.int64


## ARCTrainingComparator

In [ ]:
class ARCTrainingComparator:
    def __init__(self, train_loader, eval_loader, device="cuda", log_dir=None, checkpoint_dir="checkpoints"):
        self.train_loader = train_loader
        self.eval_loader = eval_loader
        self.device = device
        self.log_dir = log_dir or "runs/arc_training"
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        self.models = {}
        self.optimizers = {}
        self.schedulers = {}
        self.training_history = {}
        self.writer = SummaryWriter(log_dir=self.log_dir)

    def setup_models(self, configs):
        for name, config in configs.items():
            model_config = config.copy()
            for key in ["learning_rate", "weight_decay", "epochs"]:
                model_config.pop(key, None)
            if "forward_dtype" not in model_config:
                model_config["forward_dtype"] = "float32"

            if model_config.get("use_mamba2", False):
                if self.device != "cuda" and not torch.cuda.is_available():
                    raise RuntimeError("Mamba2 requires CUDA but CUDA is not available.")

            is_bidirectional = "bidirectional" in name.lower() or "bidir" in name.lower()
            if is_bidirectional:
                model = MambaBiDirectionalModel(model_config)
            else:
                model = TinyRecursiveReasoningModel_ACTV1(model_config)
            model.to(self.device)

            total_params = sum(p.numel() for p in model.parameters())
            trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            param_size_mb = total_params * 4 / (1024 ** 2)

            print(f"\n{name}:")
            print(f"  Parameters: {total_params:,} (trainable: {trainable_params:,})")
            print(f"  Model size: {param_size_mb:.2f} MB")
            if self.device == "cuda":
                torch.cuda.empty_cache()
                memory_allocated = torch.cuda.memory_allocated() / (1024 ** 2)
                memory_reserved = torch.cuda.memory_reserved() / (1024 ** 2)
                print(f"  GPU memory: {memory_allocated:.2f} MB allocated, {memory_reserved:.2f} MB reserved")

            base_lr = config.get("learning_rate", 1e-4)
            optimizer = optim.AdamW(
                model.parameters(),
                lr=base_lr,
                weight_decay=config.get("weight_decay", 0.1),
                eps=1e-8
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=config.get("epochs", 100)
            )

            self.models[name] = model
            self.optimizers[name] = optimizer
            self.schedulers[name] = scheduler
            self.training_history[name] = {
                'train_loss': [], 'train_accuracy': [], 'train_perplexity': [],
                'eval_loss': [], 'eval_accuracy': [], 'eval_perplexity': [],
                'halt_steps': [], 'epoch_time': []
            }

    def save_checkpoint(self, model_name, model, optimizer, scheduler, epoch, metrics):
        checkpoint_path = os.path.join(self.checkpoint_dir, f"{model_name}_epoch_{epoch+1}.pt")
        config_dict = model.config.__dict__ if hasattr(model.config, '__dict__') else model.config
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'metrics': metrics,
            'config': config_dict
        }
        torch.save(checkpoint, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")

    def clear_memory(self, model_name=None):
        if model_name and model_name in self.models:
            del self.models[model_name]
            del self.optimizers[model_name]
            del self.schedulers[model_name]
        if self.device == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        import gc
        gc.collect()

    def compute_metrics(self, logits, labels):
        with torch.no_grad():
            batch_size = labels.shape[0]
            target_seq_len = labels.shape[1]

            # Align logits and labels along batch and sequence dimensions
            if logits.shape[0] != batch_size:
                logits = logits[:batch_size]
            if logits.shape[1] != target_seq_len:
                if logits.shape[1] > target_seq_len:
                    logits = logits[:, :target_seq_len]
                else:
                    pad_size = target_seq_len - logits.shape[1]
                    # pad on sequence dimension: (last_dim_pad_left, last_dim_pad_right, seq_pad_left, seq_pad_right)
                    logits = F.pad(logits, (0, 0, 0, pad_size))

            # Predictions & mask
            predictions = torch.argmax(logits, dim=-1)
            mask = labels != IGNORE_LABEL_ID

            total_tokens = mask.sum().item()
            if total_tokens == 0:
                return {
                    "accuracy": 0.0,          # strict per-task
                    "soft_accuracy": 0.0,     # per-token
                    "loss": float('inf'),
                    "perplexity": float('inf')
                }

            # --- STRICT PER-TASK ACCURACY (all non-ignored tokens must match) ---
            task_correct = []
            for b in range(batch_size):
                batch_mask = mask[b]
                if batch_mask.sum() == 0:
                    task_correct.append(0.0)
                else:
                    batch_pred = predictions[b][batch_mask]
                    batch_label = labels[b][batch_mask]
                    # 1.0 only if all masked tokens match
                    all_correct = (batch_pred == batch_label).all().item()
                    task_correct.append(float(all_correct))

            accuracy = sum(task_correct) / len(task_correct) if task_correct else 0.0

            # --- SOFT PER-TOKEN ACCURACY ---
            token_correct = ((predictions == labels) & mask).sum().item()
            soft_accuracy = token_correct / max(total_tokens, 1)

            # Stable loss / perplexity with clamping
            logits_clamped = torch.clamp(logits, min=-50.0, max=50.0)
            loss = nn.functional.cross_entropy(
                logits_clamped.view(-1, logits_clamped.size(-1)),
                labels.view(-1),
                ignore_index=IGNORE_LABEL_ID,
                reduction='mean'
            ).item()
            loss = min(loss, 100.0)
            perplexity = min(torch.exp(torch.tensor(loss)).item(), 1e10)

            return {
                "accuracy":      accuracy,       # strict per-task
                "soft_accuracy": soft_accuracy,  # per-token
                "loss":          loss,
                "perplexity":    perplexity
            }



    def train_epoch(self, model, optimizer, model_name, epoch=0, gradient_accumulation_steps=1):
        model.train()
        epoch_start = time.time()
        total_loss = 0
        total_accuracy = 0
        total_soft_accuracy = 0
        total_perplexity = 0
        total_halt_steps = 0
        num_batches = 0
        accumulated_loss = 0

        progress_bar = tqdm(self.train_loader, desc=f"Training {model_name}")

        for batch_idx, batch in enumerate(progress_bar):
            batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}

            carry = model.initial_carry(batch)
            all_logits = []
            halt_steps_list = []

            for step in range(model.config.halt_max_steps):
                carry, outputs = model(carry, batch)
                if torch.isnan(outputs["logits"]).any() or torch.isinf(outputs["logits"]).any():
                    break
                all_logits.append(outputs["logits"])
                if isinstance(carry, tuple):
                    halt_steps_list.append(carry[0].steps.float().mean().item())
                    if carry[0].halted.all() and carry[1].halted.all():
                        break
                else:
                    halt_steps_list.append(carry.steps.float().mean().item())
                    if carry.halted.all():
                        break

            if len(all_logits) == 0:
                del batch, carry
                continue

            logits = torch.stack(all_logits).mean(0)
            if torch.isnan(logits).any() or torch.isinf(logits).any():
                del batch, carry, logits
                continue

            vocab_size = model.config.vocab_size
            if logits.size(-1) == 2 * vocab_size:
                logits_combined = logits[:, :, :vocab_size] + logits[:, :, vocab_size:]
            else:
                logits_combined = logits

            loss = nn.functional.cross_entropy(
                logits_combined.reshape(-1, logits_combined.size(-1)),
                batch["labels"].reshape(-1),
                ignore_index=IGNORE_LABEL_ID,
                reduction='mean'
            ) / gradient_accumulation_steps

            if torch.isnan(loss) or torch.isinf(loss):
                del batch, carry, logits, loss
                continue

            loss.backward()
            accumulated_loss += loss.item()

            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                max_grad_norm = 0.5 if "mamba" in model_name.lower() else 1.0
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

                has_nan_grad = False
                for param in model.parameters():
                    if param.grad is not None:
                        if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
                            has_nan_grad = True
                            break

                if not (has_nan_grad or torch.isnan(grad_norm) or torch.isinf(grad_norm)):
                    optimizer.step()
                optimizer.zero_grad()

                metrics = self.compute_metrics(logits, batch["labels"])

                total_loss += metrics["loss"]
                total_accuracy += metrics["accuracy"]
                total_soft_accuracy += metrics["soft_accuracy"]
                total_perplexity += metrics["perplexity"]

                total_halt_steps += np.mean(halt_steps_list)
                num_batches += 1
                accumulated_loss = 0

                global_step = epoch * len(self.train_loader) + batch_idx
                self.writer.add_scalar(f'{model_name}/train/batch_loss', metrics["loss"], global_step)
                self.writer.add_scalar(f'{model_name}/train/batch_accuracy', metrics["accuracy"], global_step)
                self.writer.add_scalar(f'{model_name}/train/batch_soft_accuracy', metrics["soft_accuracy"], global_step)
                self.writer.add_scalar(f'{model_name}/train/batch_halt_steps', np.mean(halt_steps_list), global_step)


                if batch_idx % 10 == 0:
                    progress_bar.set_postfix({
                        "loss": f"{metrics['loss']:.4f}",
                        "acc": f"{metrics['accuracy']:.4f}",
                        "halt": f"{np.mean(halt_steps_list):.1f}"
                    })

            del batch, carry, logits, loss
            if self.device == "cuda" and batch_idx % 50 == 0:
                torch.cuda.empty_cache()

        epoch_time = time.time() - epoch_start
        if num_batches == 0:
            return {
                "train_loss": float('nan'),
                "train_accuracy": 0.0,
                "train_perplexity": float('nan'),
                "halt_steps": 0.0,
                "epoch_time": epoch_time
            }

        num_batches_safe = max(num_batches, 1)
        return {
            "train_loss": total_loss / num_batches_safe,
            "train_accuracy": total_accuracy / num_batches_safe,
            "soft_train_accuracy": total_soft_accuracy / num_batches_safe,
            "train_perplexity": total_perplexity / num_batches_safe,
            "halt_steps": total_halt_steps / num_batches_safe,
            "epoch_time": epoch_time
        }


    def evaluate(self, model, model_name):
        model.eval()
        total_loss = 0
        total_accuracy = 0
        total_soft_accuracy = 0
        total_perplexity = 0
        total_halt_steps = 0
        num_batches = 0

        with torch.no_grad():
            for batch_idx, batch in enumerate(tqdm(self.eval_loader, desc=f"Evaluating {model_name}")):
                batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}
                carry = model.initial_carry(batch)
                all_logits = []
                halt_steps_list = []

                for step in range(model.config.halt_max_steps):
                    carry, outputs = model(carry, batch)
                    all_logits.append(outputs["logits"])
                    if isinstance(carry, tuple):
                        halt_steps_list.append(carry[0].steps.float().mean().item())
                        if carry[0].halted.all() and carry[1].halted.all():
                            break
                    else:
                        halt_steps_list.append(carry.steps.float().mean().item())
                        if carry.halted.all():
                            break

                logits = torch.stack(all_logits).mean(0)
                vocab_size = model.config.vocab_size
                if logits.size(-1) == 2 * vocab_size:
                    logits_combined = logits[:, :, :vocab_size] + logits[:, :, vocab_size:]
                else:
                    logits_combined = logits
                metrics = self.compute_metrics(logits_combined, batch["labels"])

                total_loss += metrics["loss"]
                total_accuracy += metrics["accuracy"]
                total_soft_accuracy += metrics["soft_accuracy"]
                total_perplexity += metrics["perplexity"]

                total_halt_steps += np.mean(halt_steps_list)
                num_batches += 1

                del batch, carry, logits
                if self.device == "cuda" and batch_idx % 50 == 0:
                    torch.cuda.empty_cache()

        if num_batches == 0:
            return {
                "eval_loss": float('nan'),
                "eval_accuracy": 0.0,
                "eval_perplexity": float('nan'),
                "eval_halt_steps": 0.0
            }

        num_batches_safe = max(num_batches, 1)
        return {
            "eval_loss": total_loss / num_batches_safe,
            "eval_accuracy": total_accuracy / num_batches_safe,
            "soft_eval_accuracy": total_soft_accuracy / num_batches_safe,
            "eval_perplexity": total_perplexity / num_batches_safe,
            "eval_halt_steps": total_halt_steps / num_batches_safe
        }


    def train_single_model(self, model_name, epochs, gradient_accumulation_steps=1):
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} not found. Available: {list(self.models.keys())}")

        model = self.models[model_name]
        optimizer = self.optimizers[model_name]
        scheduler = self.schedulers[model_name]

        print(f"\n{'='*80}")
        print(f"TRAINING {model_name.upper()} - {epochs} EPOCHS")
        print(f"{'='*80}")

        for epoch in range(epochs):
            print(f"\n{'='*80}")
            print(f"EPOCH {epoch + 1}/{epochs} - {model_name.upper()}")
            print(f"{'='*80}")

            train_metrics = self.train_epoch(
                model,
                optimizer,
                model_name,
                epoch,
                gradient_accumulation_steps=gradient_accumulation_steps
            )
            scheduler.step()
            eval_metrics = self.evaluate(model, model_name)

            all_metrics = {**train_metrics, **eval_metrics}

            for key, value in all_metrics.items():
                if key not in self.training_history[model_name]:
                    self.training_history[model_name][key] = []
                self.training_history[model_name][key].append(value)

            self.writer.add_scalar(f'{model_name}/train/loss', train_metrics['train_loss'], epoch)
            self.writer.add_scalar(f'{model_name}/train/accuracy', train_metrics['train_accuracy'], epoch)
            self.writer.add_scalar(f'{model_name}/train/soft_accuracy', train_metrics['soft_train_accuracy'], epoch)
            self.writer.add_scalar(f'{model_name}/train/perplexity', train_metrics['train_perplexity'], epoch)
            self.writer.add_scalar(f'{model_name}/train/halt_steps', train_metrics['halt_steps'], epoch)

            self.writer.add_scalar(f'{model_name}/eval/loss', eval_metrics['eval_loss'], epoch)
            self.writer.add_scalar(f'{model_name}/eval/accuracy', eval_metrics['eval_accuracy'], epoch)
            self.writer.add_scalar(f'{model_name}/eval/soft_accuracy', eval_metrics['soft_eval_accuracy'], epoch)
            self.writer.add_scalar(f'{model_name}/eval/perplexity', eval_metrics['eval_perplexity'], epoch)
            self.writer.add_scalar(f'{model_name}/eval/halt_steps', eval_metrics['eval_halt_steps'], epoch)

            self.writer.add_scalar(f'{model_name}/time/epoch_time', train_metrics['epoch_time'], epoch)
            self.writer.add_scalar(f'{model_name}/lr', optimizer.param_groups[0]['lr'], epoch)

            print(f"Train Loss: {train_metrics['train_loss']:.4f} | "
                  f"Train Acc: {train_metrics['train_accuracy']:.4f} | "
                  f"Train SoftAcc: {train_metrics['soft_train_accuracy']:.4f} | "
                  f"Eval Acc: {eval_metrics['eval_accuracy']:.4f} | "
                  f"Eval SoftAcc: {eval_metrics['soft_eval_accuracy']:.4f} | "
                  f"Halt Steps: {train_metrics['halt_steps']:.2f} | "
                  f"Time: {train_metrics['epoch_time']:.2f}s")


            self.save_checkpoint(model_name, model, optimizer, scheduler, epoch, all_metrics)

            if self.device == "cuda":
                torch.cuda.empty_cache()

        print(f"\n{model_name.upper()} training completed!")
        self.writer.flush()

    def train_comparison(self, epochs=50, gradient_accumulation_steps=1):
        model_names = list(self.models.keys())

        for model_name in model_names:
            self.train_single_model(model_name, epochs, gradient_accumulation_steps)
            print(f"\nClearing memory after {model_name} training...")
            self.clear_memory(model_name)
            print(f"Memory cleared. GPU memory: {torch.cuda.memory_allocated() / (1024**2):.2f} MB" if self.device == "cuda" else "Memory cleared.")

        self._final_analysis()

    def _print_epoch_comparison(self, results, epoch):
        print(f"\n{'-'*100}")
        print(f"COMPARISON - Epoch {epoch + 1}")
        print(f"{'-'*100}")
        print(f"{'Model':<20} {'Train Acc':<12} {'Eval Acc':<12} {'Eval Loss':<12} {'Halt Steps':<12} {'Time (s)':<12}")
        print(f"{'-'*100}")

        for model_name, metrics in results.items():
            print(f"{model_name:<20} "
                  f"{metrics['train_accuracy']:<12.4f} "
                  f"{metrics['eval_accuracy']:<12.4f} "
                  f"{metrics['eval_loss']:<12.4f} "
                  f"{metrics['halt_steps']:<12.2f} "
                  f"{metrics['epoch_time']:<12.2f}")

    def _final_analysis(self):
        print(f"\n{'='*120}")
        print("FINAL COMPARISON ANALYSIS")
        print(f"{'='*120}")

        analysis_data = []
        for model_name, history in self.training_history.items():
            if not history['eval_accuracy']:
                continue
            eval_acc_valid = [x for x in history['eval_accuracy'] if not (np.isnan(x) or np.isinf(x))]
            if not eval_acc_valid:
                continue

            final_accuracy = history['eval_accuracy'][-1]
            if np.isnan(final_accuracy) or np.isinf(final_accuracy):
                final_accuracy = eval_acc_valid[-1] if eval_acc_valid else 0.0

            best_accuracy = max(eval_acc_valid)
            convergence_epoch = np.argmax(history['eval_accuracy']) + 1
            avg_epoch_time = np.mean([x for x in history['epoch_time'] if not (np.isnan(x) or np.isinf(x))]) if history['epoch_time'] else 0.0
            avg_halt_steps = np.mean([x for x in history['halt_steps'] if not (np.isnan(x) or np.isinf(x))]) if history['halt_steps'] else 0.0

            analysis_data.append({
                'Model': model_name,
                'Final Accuracy': final_accuracy,
                'Best Accuracy': best_accuracy,
                'Convergence Epoch': convergence_epoch,
                'Avg Epoch Time (s)': avg_epoch_time,
                'Avg Halt Steps': avg_halt_steps
            })

        if not analysis_data:
            return

        df = pd.DataFrame(analysis_data)
        print("\nPERFORMANCE SUMMARY:")
        print(f"{'-'*100}")
        print(f"{'Model':<20} {'Final Acc':<12} {'Best Acc':<12} {'Conv Epoch':<12} {'Epoch Time':<12} {'Halt Steps':<12}")
        print(f"{'-'*100}")

        for _, row in df.iterrows():
            print(f"{row['Model']:<20} "
                  f"{row['Final Accuracy']:<12.4f} "
                  f"{row['Best Accuracy']:<12.4f} "
                  f"{row['Convergence Epoch']:<12} "
                  f"{row['Avg Epoch Time (s)']:<12.2f} "
                  f"{row['Avg Halt Steps']:<12.2f}")

        best_acc_valid = df['Best Accuracy'].dropna()
        if not best_acc_valid.empty:
            best_overall = df.loc[best_acc_valid.idxmax()]
            print(f"\nBest Overall: {best_overall['Model']} (Accuracy: {best_overall['Best Accuracy']:.4f})")

        self.writer.close()


## Quick tests

In [ ]:
if doARCAGI:
    if doQuickTests:
        def quick_test_models():
            test_configs = {
                "transformer_test": {
                    "batch_size": 1,
                    "seq_len": 1800,
                    "num_puzzle_identifiers": 100,
                    "vocab_size": 12,
                    "H_cycles": 2,
                    "L_cycles": 1,
                    "H_layers": 2,
                    "L_layers": 1,
                    "hidden_size": 256,
                    "expansion": 2.0,
                    "num_heads": 4,
                    "pos_encodings": "rope",
                    "halt_max_steps": 4,
                    "halt_exploration_prob": 0.1,
                    "use_mamba2": False,
                    "mamba_d_state": 16,
                    "mamba_d_conv": 4,
                    "puzzle_emb_ndim": 128,
                    "forward_dtype": "float32",
                    "no_ACT_continue": True,
                    "learning_rate": 1e-4,
                    "weight_decay": 0.1,
                    "epochs": 2
                },
                "mamba_test": {
                    "batch_size": 1,
                    "seq_len": 1800,
                    "num_puzzle_identifiers": 100,
                    "vocab_size": 12,
                    "H_cycles": 2,
                    "L_cycles": 1,
                    "H_layers": 3,
                    "L_layers": 2,
                    "hidden_size": 256,
                    "expansion": 2.0,
                    "num_heads": 4,
                    "pos_encodings": "rope",
                    "halt_max_steps": 4,
                    "halt_exploration_prob": 0.1,
                    "use_mamba2": True,
                    "mamba_d_state": 16,
                    "mamba_d_conv": 4,
                    "puzzle_emb_ndim": 128,
                    "forward_dtype": "float32",
                    "no_ACT_continue": True,
                    "learning_rate": 1e-4,
                    "weight_decay": 0.1,
                    "epochs": 2
                }
            }

            train_set_test, eval_set_test = arckit.load_data("kaggle")
            train_loader_test, eval_loader_test = create_arc_reasoning_dataloaders(
                train_set_test[:20],
                eval_set_test[:10],
                batch_size=2,
                seq_len=1800
            )

            device = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"Quick test on {device} with {len(train_loader_test)} train batches, {len(eval_loader_test)} eval batches")

            test_comparator = ARCTrainingComparator(train_loader_test, eval_loader_test, device, log_dir="runs/quick_test")
            test_comparator.setup_models(test_configs)
            test_comparator.train_comparison(epochs=2)

            print("\nQuick test completed successfully!")
            return True

## Full

In [ ]:
if doARCAGI:
    if doFullTests:
        transformer_training_config = {
            "transformer": {
                **transformer_config,
                "learning_rate": 3e-4,
                "weight_decay": 0.1,
                "epochs": 10
            }
        }

        mamba_training_config = {
            "mamba": {
                **mamba_config,
                "learning_rate": 3e-4,
                "weight_decay": 0.1,
                "epochs": 10
            }
        }

        mamba_bidirectional_training_config = {
            "mamba_bidirectional": {
                **mamba_bidirectional_config,
                "learning_rate": 3e-4,
                "weight_decay": 0.1,
                "epochs": 10
            }
        }

        train_set, eval_set = arckit.load_data("kaggle")
        train_loader, eval_loader = create_arc_reasoning_dataloaders(
            train_set,
            eval_set,
            batch_size=1,
            seq_len=3800
        )

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Device: {device}")
        print(f"Train batches: {len(train_loader)}, Eval batches: {len(eval_loader)}")

Device: cuda
Train batches: 1000, Eval batches: 120


In [ ]:
if doARCAGI:
    if doFullTests:
        print("="*80)
        print("TRAINING MAMBA BIDIRECTIONAL MODEL")
        print("="*80)

        mamba_bidirectional_comparator = ARCTrainingComparator(
            train_loader, eval_loader, device,
            log_dir="runs/mamba_bidirectional_training",
            checkpoint_dir="checkpoints/mamba_bidirectional"
        )
        mamba_bidirectional_comparator.setup_models(mamba_bidirectional_training_config)
        mamba_bidirectional_comparator.train_single_model("mamba_bidirectional", epochs=10, gradient_accumulation_steps=2)
        mamba_bidirectional_comparator.save_checkpoint("mamba_bidirectional",
            mamba_bidirectional_comparator.models["mamba_bidirectional"],
            mamba_bidirectional_comparator.optimizers["mamba_bidirectional"],
            mamba_bidirectional_comparator.schedulers["mamba_bidirectional"],
            9, {})
        print("\nClearing memory after mamba bidirectional training...")
        mamba_bidirectional_comparator.clear_memory("mamba_bidirectional")
        if device == "cuda":
            print(f"GPU memory after clearing: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")
        mamba_bidirectional_comparator.writer.close()
        print(f"\nMamba bidirectional training completed!")
        print(f"Checkpoints saved to: {mamba_bidirectional_comparator.checkpoint_dir}")
        print(f"TensorBoard logs saved to: {mamba_bidirectional_comparator.log_dir}")


TRAINING MAMBA BIDIRECTIONAL MODEL

mamba_bidirectional:
  Parameters: 6,236,164 (trainable: 6,236,164)
  Model size: 23.79 MB
  GPU memory: 29.04 MB allocated, 30.00 MB reserved

TRAINING MAMBA_BIDIRECTIONAL - 10 EPOCHS

EPOCH 1/10 - MAMBA_BIDIRECTIONAL


Training mamba_bidirectional:   1%|▏         | 13/1000 [00:41<52:15,  3.18s/it] 


KeyboardInterrupt: 

In [ ]:
if doARCAGI:
    if doFullTests:
        print("="*80)
        print("TRAINING TRANSFORMER MODEL")
        print("="*80)

        transformer_comparator = ARCTrainingComparator(
            train_loader, eval_loader, device,
            log_dir="runs/transformer_training",
            checkpoint_dir="checkpoints/transformer"
        )
        transformer_comparator.setup_models(transformer_training_config)
        transformer_comparator.train_single_model("transformer", epochs=10, gradient_accumulation_steps=2)
        transformer_comparator.save_checkpoint("transformer",
            transformer_comparator.models["transformer"],
            transformer_comparator.optimizers["transformer"],
            transformer_comparator.schedulers["transformer"],
            9, {})
        print("\nClearing memory after transformer training...")
        transformer_comparator.clear_memory("transformer")
        if device == "cuda":
            print(f"GPU memory after clearing: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")
        transformer_comparator.writer.close()
        print(f"\nTransformer training completed!")
        print(f"Checkpoints saved to: {transformer_comparator.checkpoint_dir}")
        print(f"TensorBoard logs saved to: {transformer_comparator.log_dir}")


# SUDOKU

## Preprocessing Sudoku

In [ ]:
import os
import csv
import json
import numpy as np
from huggingface_hub import hf_hub_download
from tqdm import tqdm

SUDOKU_OUTPUT_DIR = "data/sudoku-extreme-full"


def shuffle_sudoku(board: np.ndarray, solution: np.ndarray):
    digit_map = np.pad(np.random.permutation(np.arange(1, 10)), (1, 0))

    transpose_flag = np.random.rand() < 0.5

    bands = np.random.permutation(3)
    row_perm = np.concatenate([b * 3 + np.random.permutation(3) for b in bands])

    stacks = np.random.permutation(3)
    col_perm = np.concatenate([s * 3 + np.random.permutation(3) for s in stacks])

    mapping = np.array([row_perm[i // 9] * 9 + col_perm[i % 9] for i in range(81)])

    def apply_transformation(x: np.ndarray) -> np.ndarray:
        if transpose_flag:
            x = x.T
        new_board = x.flatten()[mapping].reshape(9, 9).copy()
        return digit_map[new_board]

    return apply_transformation(board), apply_transformation(solution)


def convert_subset_sudoku(
    set_name: str,
    source_repo: str = "sapientinc/sudoku-extreme",
    output_dir: str = SUDOKU_OUTPUT_DIR,
    subsample_size: int | None = None,
    min_difficulty: int | None = None,
    num_aug: int = 0,
):
    inputs = []
    labels = []

    csv_path = hf_hub_download(source_repo, f"{set_name}.csv", repo_type="dataset")
    with open(csv_path, newline="") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for source, q, a, rating in reader:
            if (min_difficulty is None) or (int(rating) >= min_difficulty):
                assert len(q) == 81 and len(a) == 81
                inputs.append(
                    np.frombuffer(q.replace('.', '0').encode(), dtype=np.uint8)
                    .reshape(9, 9) - ord('0')
                )
                labels.append(
                    np.frombuffer(a.encode(), dtype=np.uint8)
                    .reshape(9, 9) - ord('0')
                )

    if set_name == "train" and subsample_size is not None:
        total_samples = len(inputs)
        if subsample_size < total_samples:
            indices = np.random.choice(total_samples, size=subsample_size, replace=False)
            inputs = [inputs[i] for i in indices]
            labels = [labels[i] for i in indices]

    num_augments = num_aug if set_name == "train" else 0

    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    puzzle_id = 0
    example_id = 0

    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)

    for orig_inp, orig_out in zip(tqdm(inputs, desc=f"Building {set_name}"), labels):
        for aug_idx in range(1 + num_augments):
            if aug_idx == 0:
                inp, out = orig_inp, orig_out
            else:
                inp, out = shuffle_sudoku(orig_inp, orig_out)

            results["inputs"].append(inp)
            results["labels"].append(out)
            example_id += 1
            puzzle_id += 1

            results["puzzle_indices"].append(example_id)
            results["puzzle_identifiers"].append(0)

        results["group_indices"].append(puzzle_id)

    def _seq_to_numpy(seq):
        arr = np.concatenate(seq).reshape(len(seq), -1)
        assert np.all((arr >= 0) & (arr <= 9))
        return arr + 1

    results = {
        "inputs": _seq_to_numpy(results["inputs"]),
        "labels": _seq_to_numpy(results["labels"]),
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }

    metadata = {
        "seq_len": 81,
        "vocab_size": 11,
        "pad_id": 0,
        "ignore_label_id": 0,
        "blank_identifier_id": 0,
        "num_puzzle_identifiers": 1,
        "total_groups": len(results["group_indices"]) - 1,
        "mean_puzzle_examples": 1,
        "total_puzzles": len(results["group_indices"]) - 1,
        "sets": ["all"],
    }

    save_dir = os.path.join(output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)

    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata, f)

    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)

    with open(os.path.join(output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)


def preprocess_sudoku_full(
    source_repo: str = "sapientinc/sudoku-extreme",
    output_dir: str = SUDOKU_OUTPUT_DIR,
    subsample_size: int | None = None,
    min_difficulty: int | None = None,
    num_aug: int = 0,
):
    convert_subset_sudoku("train", source_repo, output_dir, subsample_size, min_difficulty, num_aug)
    convert_subset_sudoku("test", source_repo, output_dir, subsample_size=None, min_difficulty=min_difficulty, num_aug=0)


In [ ]:
if doSudoku:
    preprocess_sudoku_full(
        source_repo="sapientinc/sudoku-extreme",
        output_dir=SUDOKU_OUTPUT_DIR,
        subsample_size=None,
        min_difficulty=None,
        num_aug=0,
    )


train.csv:   0%|          | 0.00/719M [00:00<?, ?B/s]

Building train: 100%|██████████| 3831994/3831994 [00:02<00:00, 1891165.19it/s]


test.csv:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Building test: 100%|██████████| 422786/422786 [00:00<00:00, 1807710.00it/s]


## SudokuDataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SudokuDataset(Dataset):
    def __init__(
        self,
        inputs: np.ndarray,
        labels: np.ndarray,
        puzzle_identifiers: np.ndarray,
        seq_len: int = 81,
        pad_token_id: int = 0,
    ):
        assert inputs.shape == labels.shape
        assert inputs.ndim == 2
        self.inputs = inputs
        self.labels = labels
        self.puzzle_identifiers = puzzle_identifiers
        self.seq_len = seq_len
        self.pad_token_id = pad_token_id

    def __len__(self):
        return self.inputs.shape[0]

    def __getitem__(self, idx):
        inp_np = self.inputs[idx]
        tgt_np = self.labels[idx]

        inp = torch.from_numpy(inp_np).view(-1).long()
        tgt = torch.from_numpy(tgt_np).view(-1).long()

        seq_len = self.seq_len

        fixed_inputs = torch.full(
            (seq_len,),
            self.pad_token_id,
            dtype=torch.long
        )
        fixed_targets = torch.full(
            (seq_len,),
            IGNORE_LABEL_ID,
            dtype=torch.long
        )

        L = min(seq_len, inp.size(0))
        fixed_inputs[:L] = inp[:L]
        fixed_targets[:L] = tgt[:L]

        return {
            "inputs": fixed_inputs,
            "targets": fixed_targets,
            "labels": fixed_targets.clone(),
            "puzzle_identifiers": torch.tensor(
                self.puzzle_identifiers[idx],
                dtype=torch.long
            ),
        }


def load_sudoku_split(split_dir: str):
    inputs = np.load(os.path.join(split_dir, "all__inputs.npy"))
    labels = np.load(os.path.join(split_dir, "all__labels.npy"))
    puzzle_identifiers = np.load(os.path.join(split_dir, "all__puzzle_identifiers.npy"))

    with open(os.path.join(split_dir, "dataset.json"), "r") as f:
        metadata = json.load(f)

    return inputs, labels, puzzle_identifiers, metadata

from torch.utils.data import DataLoader, Subset

from torch.utils.data import DataLoader
import numpy as np

def create_sudoku_dataloaders(
    train_dir: str,
    eval_dir: str,
    batch_size: int = 32,
    num_workers: int = 0,
    max_train_samples: int | None = None,
    max_eval_samples: int | None = None,
):
    train_inputs, train_labels, train_pids, train_meta = load_sudoku_split(train_dir)
    eval_inputs, eval_labels, eval_pids, eval_meta = load_sudoku_split(eval_dir)

    seq_len = train_meta.get("seq_len", 81)
    pad_id = train_meta.get("pad_id", 0)

    if max_train_samples is not None and max_train_samples < train_inputs.shape[0]:
        idx = np.random.permutation(train_inputs.shape[0])[:max_train_samples]
        train_inputs = train_inputs[idx]
        train_labels = train_labels[idx]
        train_pids   = train_pids[idx]

    if max_eval_samples is not None and max_eval_samples < eval_inputs.shape[0]:
        idx = np.random.permutation(eval_inputs.shape[0])[:max_eval_samples]
        eval_inputs = eval_inputs[idx]
        eval_labels = eval_labels[idx]
        eval_pids   = eval_pids[idx]

    train_dataset = SudokuDataset(
        train_inputs,
        train_labels,
        train_pids,
        seq_len=seq_len,
        pad_token_id=pad_id,
    )
    eval_dataset = SudokuDataset(
        eval_inputs,
        eval_labels,
        eval_pids,
        seq_len=seq_len,
        pad_token_id=pad_id,
    )

    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, eval_loader


## Full tests

In [ ]:
if doSudoku:
    if doFullTests:
        SUDOKU_SEQ_LEN = 81
        SUDOKU_VOCAB_SIZE = 11
        SUDOKU_NUM_PIDS = 1

        SUDOKU_MAX_TRAIN_SAMPLES = 1_000
        SUDOKU_MAX_EVAL_SAMPLES  = 256

        SUDOKU_BATCH_SIZE = 2

        SUDOKU_EPOCHS = 10

        SUDOKU_LR = 3e-4
        SUDOKU_WD = 0.1
        SUDOKU_GRAD_ACC_STEPS = 1

        sudoku_transformer_training_config = {
            "transformer": {
                **transformer_config,
                "batch_size": SUDOKU_BATCH_SIZE,
                "seq_len": SUDOKU_SEQ_LEN,
                "num_puzzle_identifiers": SUDOKU_NUM_PIDS,
                "vocab_size": SUDOKU_VOCAB_SIZE,
                "learning_rate": SUDOKU_LR,
                "weight_decay": SUDOKU_WD,
                "epochs": SUDOKU_EPOCHS,
            }
        }

        # 🔁 bidirectional mamba config (replaces old sudoku_mamba_training_config)
        sudoku_mamba_bidirectional_training_config = {
            "mamba_bidirectional": {
                **mamba_bidirectional_config,
                "batch_size": SUDOKU_BATCH_SIZE,
                "seq_len": SUDOKU_SEQ_LEN,
                "num_puzzle_identifiers": SUDOKU_NUM_PIDS,
                "vocab_size": SUDOKU_VOCAB_SIZE,
                "learning_rate": SUDOKU_LR,
                "weight_decay": SUDOKU_WD,
                "epochs": SUDOKU_EPOCHS,
            }
        }

        sudoku_train_dir = os.path.join(SUDOKU_OUTPUT_DIR, "train")
        sudoku_eval_dir  = os.path.join(SUDOKU_OUTPUT_DIR, "test")

        sudoku_train_loader, sudoku_eval_loader = create_sudoku_dataloaders(
            train_dir=sudoku_train_dir,
            eval_dir=sudoku_eval_dir,
            batch_size=SUDOKU_BATCH_SIZE,
            num_workers=4,
            max_train_samples=SUDOKU_MAX_TRAIN_SAMPLES,
            max_eval_samples=SUDOKU_MAX_EVAL_SAMPLES,
        )

        sudoku_device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Sudoku device: {sudoku_device}")
        print(f"Sudoku train batches: {len(sudoku_train_loader)}, eval batches: {len(sudoku_eval_loader)}")

        # ============================
        #  BIDIRECTIONAL MAMBA ON SUDOKU
        # ============================
        print("=" * 80)
        print("TRAINING BIDIRECTIONAL MAMBA MODEL ON SUDOKU SUBSET")
        print("=" * 80)

        sudoku_mamba_bidir_comparator = ARCTrainingComparator(
            sudoku_train_loader,
            sudoku_eval_loader,
            sudoku_device,
            log_dir="runs/sudoku_mamba_bidirectional_full",
            checkpoint_dir="checkpoints/sudoku_mamba_bidirectional_full",
        )
        sudoku_mamba_bidir_comparator.setup_models(sudoku_mamba_bidirectional_training_config)

        mamba_bidir_epochs = sudoku_mamba_bidirectional_training_config["mamba_bidirectional"]["epochs"]
        sudoku_mamba_bidir_comparator.train_single_model(
            "mamba_bidirectional",
            epochs=mamba_bidir_epochs,
            gradient_accumulation_steps=SUDOKU_GRAD_ACC_STEPS,
        )

        sudoku_mamba_bidir_comparator.save_checkpoint(
            "mamba_bidirectional",
            sudoku_mamba_bidir_comparator.models["mamba_bidirectional"],
            sudoku_mamba_bidir_comparator.optimizers["mamba_bidirectional"],
            sudoku_mamba_bidir_comparator.schedulers["mamba_bidirectional"],
            mamba_bidir_epochs - 1,
            {},
        )

        print("\nClearing memory after Sudoku Bidirectional Mamba training...")
        sudoku_mamba_bidir_comparator.clear_memory("mamba_bidirectional")
        if sudoku_device == "cuda":
            print(f"GPU memory after clearing: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")
        sudoku_mamba_bidir_comparator.writer.close()
        print(f"\nSudoku Bidirectional Mamba training completed!")
        print(f"Checkpoints saved to: {sudoku_mamba_bidir_comparator.checkpoint_dir}")
        print(f"TensorBoard logs saved to: {sudoku_mamba_bidir_comparator.log_dir}")

        # ============================
        #  TRANSFORMER ON SUDOKU
        # ============================
        print("=" * 80)
        print("TRAINING TRANSFORMER MODEL ON SUDOKU SUBSET")
        print("=" * 80)

        sudoku_transformer_comparator = ARCTrainingComparator(
            sudoku_train_loader,
            sudoku_eval_loader,
            sudoku_device,
            log_dir="runs/sudoku_transformer_full",
            checkpoint_dir="checkpoints/sudoku_transformer_full",
        )
        sudoku_transformer_comparator.setup_models(sudoku_transformer_training_config)

        transformer_epochs = sudoku_transformer_training_config["transformer"]["epochs"]
        sudoku_transformer_comparator.train_single_model(
            "transformer",
            epochs=transformer_epochs,
            gradient_accumulation_steps=SUDOKU_GRAD_ACC_STEPS,
        )

        sudoku_transformer_comparator.save_checkpoint(
            "transformer",
            sudoku_transformer_comparator.models["transformer"],
            sudoku_transformer_comparator.optimizers["transformer"],
            sudoku_transformer_comparator.schedulers["transformer"],
            transformer_epochs - 1,
            {},
        )

        print("\nClearing memory after Sudoku Transformer training...")
        sudoku_transformer_comparator.clear_memory("transformer")
        if sudoku_device == "cuda":
            print(f"GPU memory after clearing: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")
        sudoku_transformer_comparator.writer.close()
        print(f"\nSudoku Transformer training completed!")
        print(f"Checkpoints saved to: {sudoku_transformer_comparator.checkpoint_dir}")
        print(f"TensorBoard logs saved to: {sudoku_transformer_comparator.log_dir}")


Sudoku device: cuda
Sudoku train batches: 32, eval batches: 8
TRAINING MAMBA MODEL ON SUDOKU SUBSET

mamba:
  Parameters: 8,636,930 (trainable: 8,636,930)
  Model size: 32.95 MB
  GPU memory: 33.50 MB allocated, 42.00 MB reserved

TRAINING MAMBA - 10 EPOCHS

EPOCH 1/10 - MAMBA


Training mamba:   0%|          | 0/32 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 224.12 MiB is free. Process 2885 has 14.52 GiB memory in use. Of the allocated memory 14.13 GiB is allocated by PyTorch, and 273.50 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)